## Importing Libraries

In this step, we import the essential Python libraries required for data analysis and numerical computations.

- **pandas**: used for data manipulation and working with DataFrames  
- **numpy**: used for numerical operations and handling arrays

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import shapiro
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats



## Loading the Dataset

In this step, we load the dataset from a CSV file and perform a quick initial inspection to understand its structure.

We also check:
- The shape of the dataset (number of rows and columns)
- The column names

In [ ]:
df=pd.read_csv("social_anxiety_dataset.csv")

df.shape
df.columns

Index(['Age', 'Gender', 'Occupation', 'Sleep Hours',
       'Physical Activity (hrs/week)', 'Caffeine Intake (mg/day)',
       'Alcohol Consumption (drinks/week)', 'Smoking',
       'Family History of Anxiety', 'Stress Level (1-10)', 'Heart Rate (bpm)',
       'Breathing Rate (breaths/min)', 'Sweating Level (1-5)', 'Dizziness',
       'Medication', 'Therapy Sessions (per month)', 'Recent Major Life Event',
       'Diet Quality (1-10)', 'Anxiety Level (1-10)', 'Target', 'is_Anxious',
       'Therapy History'],
      dtype='str')

In [ ]:
df["is_Anxious"].sum()

np.int64(210)

## Checking Missing Values

In this step, we check for missing values in each column of the dataset.

This helps us understand:
- Which features contain null values
- The extent of missing data in each column
- Which columns may require cleaning or imputation



In [ ]:
df.isnull().sum()
#Age,Gender,sleep,caffeine,smoking,medication,diet,therapy

Age                                    62
Gender                                119
Occupation                              0
Sleep Hours                            36
Physical Activity (hrs/week)            0
Caffeine Intake (mg/day)               94
Alcohol Consumption (drinks/week)       0
Smoking                               106
Family History of Anxiety               0
Stress Level (1-10)                     0
Heart Rate (bpm)                        0
Breathing Rate (breaths/min)            0
Sweating Level (1-5)                    0
Dizziness                               0
Medication                            109
Therapy Sessions (per month)            0
Recent Major Life Event                 0
Diet Quality (1-10)                    78
Anxiety Level (1-10)                    0
Target                                  0
is_Anxious                              0
Therapy History                      1827
dtype: int64

In [ ]:
df.columns

Index(['Age', 'Gender', 'Occupation', 'Sleep Hours',
       'Physical Activity (hrs/week)', 'Caffeine Intake (mg/day)',
       'Alcohol Consumption (drinks/week)', 'Smoking',
       'Family History of Anxiety', 'Stress Level (1-10)', 'Heart Rate (bpm)',
       'Breathing Rate (breaths/min)', 'Sweating Level (1-5)', 'Dizziness',
       'Medication', 'Therapy Sessions (per month)', 'Recent Major Life Event',
       'Diet Quality (1-10)', 'Anxiety Level (1-10)', 'Target', 'is_Anxious',
       'Therapy History'],
      dtype='str')

In [ ]:
(df["Target"] == df["is_Anxious"]).all()

np.True_

In [ ]:
df = df.drop(columns=["is_Anxious"])

## Defining Column Groups

In this step, we categorize the dataset columns based on their data types and roles in analysis and modeling.

This helps to:
- Apply appropriate preprocessing techniques for each group
- Improve clarity and organization of the dataset
- Separate features based on their nature (numeric, ordinal, categorical, target)

### Column Categories

- **Numeric features**: Continuous numerical values used for calculations
- **Ordinal features**: Ordered categorical values with a meaningful scale
- **Categorical features**: Non-numeric labels without inherent order
- **Target variables**: Output labels for prediction tasks

In [ ]:
columns = {
    "numeric": [
        "Age",
        "Sleep Hours",
        "Physical Activity (hrs/week)",
        "Caffeine Intake (mg/day)",
        "Alcohol Consumption (drinks/week)",
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Therapy Sessions (per month)"
    ],

    "ordinal": [
        "Stress Level (1-10)",
        "Sweating Level (1-5)",
        "Diet Quality (1-10)",
        "Anxiety Level (1-10)"
    ],

    "categorical": [
        "Gender",
        "Occupation",
        "Smoking",
        "Family History of Anxiety",
        "Dizziness",
        "Medication",
        "Recent Major Life Event",
        "Therapy History"
    ],

    "target": [
        "Target"
       
    ]
}

In [ ]:
def one_numerical_plotly(data, col):
    fig = make_subplots(rows = 1, cols= 2)
    fig.add_trace(go.Histogram(x = data[col], name = 'Histogram Plot'), row=1, col=1)
    fig.update_layout( bargap = 0.05)
    fig.add_trace(go.Box(x = data[col], name = 'Box Plot'), row=1, col=2)


    fig.update_layout(title = { 'text': f"Histogram and Box plots of {col}", 'x': 0.5, 'xanchor': 'center'})

    fig.show()

In [ ]:
def numerical_eda(data, numeric_columns):

    for col in numeric_columns:
        one_numerical_plotly(data, col)

In [ ]:
numerical_eda(df, columns["numeric"])

## Fixing Missing Values in Therapy History

In this step, we clean and standardize the **Therapy History** column based on information from the **Therapy Sessions (per month)** feature.

This logic helps to:
- Fill missing values more meaningfully instead of leaving them as NaN
- Ensure consistency between related features
- Improve data quality for further analysis and modeling

### Rules Applied

- If a person has **therapy sessions > 0** but missing therapy history → label as **"Not Reported"**
- If a person has **0 therapy sessions** → label as **"No previous history"**

In [ ]:
def fix_therapy_history(df):
    df = df.copy()

    
    mask = (df["Therapy Sessions (per month)"] > 0) & (df["Therapy History"].isna())

    df.loc[mask, "Therapy History"] = "Not Reported"

   
    df.loc[df["Therapy Sessions (per month)"] == 0, "Therapy History"] = "No previous history"

    

    return df

In [ ]:
df["Target"].value_counts()

Target
0    1820
1     210
Name: count, dtype: int64

## Replacing Negative Values with Missing Values (NaN)

In this step, we handle invalid data by replacing negative values in numerical columns with `NaN`.

This is important because:
- Negative values may be invalid or unrealistic in certain features
- Replacing them with `NaN` allows proper handling in later cleaning steps (imputation or removal)
- It improves data quality and consistency

### Behavior

- If no columns are specified, the function automatically selects all numerical columns
- Any value less than 0 is replaced with `NaN`

In [ ]:
def replace_negative_with_nan(df, columns=None):
  

    df = df.copy()

    if columns is None:
        columns = df.select_dtypes(include="number").columns

    for col in columns:
        df.loc[df[col] < 0, col] = np.nan

    return df

In [ ]:
def shapiro_test_all_columns(df, columns=None):
    
    if columns is None:
        columns = df.select_dtypes(include="number").columns

    results = []

    for col in columns:
        data = df[col].dropna()

     
        if len(data) < 3:
            p_value = None
        else:
            stat, p_value = shapiro(data)

        results.append({
            "column": col,
            "p_value": p_value
        })

    return pd.DataFrame(results)

In [ ]:
result=shapiro_test_all_columns(df)
print(result)

                               column       p_value
0                                 Age  1.651636e-23
1                         Sleep Hours  2.371106e-53
2        Physical Activity (hrs/week)  2.322604e-35
3            Caffeine Intake (mg/day)  4.069686e-46
4   Alcohol Consumption (drinks/week)  2.861331e-22
5                 Stress Level (1-10)  9.929768e-28
6                    Heart Rate (bpm)  1.508059e-43
7        Breathing Rate (breaths/min)  2.074310e-26
8                Sweating Level (1-5)  1.265708e-35
9        Therapy Sessions (per month)  1.821924e-36
10                Diet Quality (1-10)  4.808843e-29
11               Anxiety Level (1-10)  4.665137e-34
12                             Target  8.378589e-65


## Handling Outliers Using IQR Method

In this step, we detect and handle outliers in numerical columns using the **Interquartile Range (IQR)** method.

Outliers are values that fall far outside the normal range of the data and can negatively affect analysis and model performance.

### Method Used (IQR)

For each numerical column:
- Q1 (25th percentile) and Q3 (75th percentile) are calculated
- IQR = Q3 - Q1
- Lower bound = Q1 − (multiplier × IQR)
- Upper bound = Q3 + (multiplier × IQR)

Values outside this range are clipped to the boundary values.

### Purpose

- Reduce the impact of extreme values
- Keep dataset stable without removing rows
- Improve robustness of statistical analysis

In [ ]:
def clean_outliers(df, cols):
    df = df.copy()
    multiplier = 1.5

    for col in cols:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - multiplier * IQR
        upper = Q3 + multiplier * IQR

        df[col] = df[col].clip(lower=lower, upper=upper)

    return df

## Cleaning Missing Values in Categorical Columns

In this step, we handle missing values in categorical features by replacing them with the most frequent value (mode).

This is important because:
- Categorical columns cannot be directly used with missing values in most models
- Filling with mode preserves the most common pattern in the data
- It avoids dropping rows and losing information

### Approach

- For each specified column:
  - Compute the mode (most frequent value)
  - Replace missing values (NaN) with that mode

In [ ]:
def clean_categorical(df, cols):
    df = df.copy()

    for col in cols:

        mode = df[col].mode()

        if not mode.empty:
            df[col] = df[col].fillna(mode[0])

    return df

In [ ]:
df=fix_therapy_history(df)
df=replace_negative_with_nan(df)
df=clean_outliers(df,columns["numeric"])
df=clean_outliers(df,columns["ordinal"])

df=clean_categorical(df,columns["categorical"])

In [ ]:
result=shapiro_test_all_columns(df)
print(result)

                               column       p_value
0                                 Age  1.651636e-23
1                         Sleep Hours  6.076694e-07
2        Physical Activity (hrs/week)  1.564963e-18
3            Caffeine Intake (mg/day)  1.391912e-18
4   Alcohol Consumption (drinks/week)  9.098087e-25
5                 Stress Level (1-10)  9.929768e-28
6                    Heart Rate (bpm)  1.004497e-23
7        Breathing Rate (breaths/min)  2.074310e-26
8                Sweating Level (1-5)  1.265708e-35
9        Therapy Sessions (per month)  3.122937e-35
10                Diet Quality (1-10)  4.808843e-29
11               Anxiety Level (1-10)  2.101975e-33
12                             Target  8.378589e-65


## Handling Missing Values in Numerical Columns

In this step, we handle missing values in numerical features using a combination of statistical imputation methods based on the distribution of each column.

This approach helps to:
- Preserve data distribution more accurately
- Avoid bias introduced by inappropriate imputation
- Improve model performance

### Strategy Used

For each numerical column:

1. A new indicator column is created:
   - `*_missing` → shows whether the original value was missing (1) or not (0)

2. Skewness is calculated to understand distribution shape:
   - If distribution is **approximately normal** (|skew| < 0.5):
     - Missing values are filled with the **mean**
   - If distribution is **skewed**:
     - Missing values are filled with the **median**

In [ ]:
def fillna_numeric(df, cols):
    df = df.copy()

    for col in cols:

        df[col + "_missing"] = df[col].isna().astype(int)

        skew = df[col].skew()

        if abs(skew) < 0.5:
            df[col] = df[col].fillna(df[col].mean())
        else:
            df[col] = df[col].fillna(df[col].median())

    return df

## Handling Missing Values in Ordinal Columns

In this step, we handle missing values in ordinal features using a simple statistical approach.

Since ordinal variables have a meaningful order (but not true numerical distance), we use the **median** for imputation.

This helps to:
- Preserve the natural ordering of the data
- Reduce the impact of extreme values compared to mean
- Maintain consistency in ranked features

### Additional Step

- A missing value indicator column (`*_missing`) is created for each feature to track originally missing values

In [ ]:
def fillna_ordinal(df, cols):
    df = df.copy()

    for col in cols:

        df[col + "_missing"] = df[col].isna().astype(int)

        df[col] = df[col].fillna(df[col].median())

    return df

In [ ]:
df=fillna_numeric(df,columns["numeric"])
df=fillna_ordinal(df,columns["ordinal"])


In [ ]:
numerical_eda(df,columns["numeric"])

## Feature Binning (Discretization)

In this step, we convert continuous numerical features into **categorical ranges (bins)**. This process is called **binning** or **discretization**.

Binning helps to:
- Simplify complex numerical data
- Capture patterns in ranges instead of exact values
- Improve interpretability of features
- Reduce sensitivity to noise and outliers

### Approach

We define custom bin ranges for each feature using domain knowledge and assign meaningful labels to each range.

Each new feature is created with the prefix:
- `Binned_ColumnName`

In [ ]:
def bining_features(df):
    df = df.copy()

    bin_config = {

        "Age": (
            [0, 18, 30, 50, 120],
            ["Teen", "Young Adult", "Adult", "Senior"]
        ),

        "Sleep Hours": (
            [0, 5, 7, 9, 24],
            ["Sleep Deprived", "Normal Sleep", "Healthy Sleep", "Oversleeping"]
        ),

        "Physical Activity (hrs/week)": (
            [-1, 2, 5, 100],
            ["Non-Athletic", "Moderately Active", "Athletic"]
        ),

        "Caffeine Intake (mg/day)": (
            [-1, 100, 300, 600, 2000],
            ["Low", "Moderate", "High", "Coffee Addict"]
        ),

        "Alcohol Consumption (drinks/week)": (
            [-1, 0, 5, 10, 100],
            ["None", "Light", "Moderate", "Heavy"]
        ),

        "Heart Rate (bpm)": (
            [0, 60, 100, 200],
            ["Low", "Normal", "High"]
        ),

        "Breathing Rate (breaths/min)": (
            [0, 12, 20, 100],
            ["Slow", "Normal", "Rapid"]
        ),

        "Therapy Sessions (per month)": (
            [-1, 0, 2, 5, 20],
            ["No Therapy", "Occasional", "Regular", "Intensive"]
        ),

        "Stress Level (1-10)": (
            [0, 3, 6, 10],
            ["Low Stress", "Moderate Stress", "High Stress"]
        ),

        "Diet Quality (1-10)": (
            [0, 3, 6, 8, 10],
            ["Unhealthy", "Average", "Healthy", "Excellent"]
        ),

        "Anxiety Level (1-10)": (
            [0, 3, 6, 10],
            ["Low Anxiety", "Moderate Anxiety", "High Anxiety"]
        ),
         "Sleep Hours": (
            [0, 4, 7, 9,24],
            ["Sleep Deprived", "Insufficient Sleep", "Recommended Sleep","Oversleeping"]
        ),
        

    }

    for col in bin_config:

        bins, labels = bin_config[col]
        df[f"Binned_{col}"] = pd.cut(
            df[col],
            bins=bins,
            labels=labels
            )

       
    return df

In [ ]:
df=bining_features(df)

## Reclassifying Categorical Features

In this step, we standardize and group categorical values into broader and more meaningful categories using a mapping dictionary.

This process helps to:
- Reduce the number of unique categories
- Improve interpretability of categorical features
- Make data more suitable for encoding and modeling
- Handle inconsistent or overly detailed labels

### Function Overview

This function:
- Replaces old category values with new grouped labels using a mapping dictionary
- Can either overwrite the original column or create a new one

In [ ]:
def reclassify_categorical(data, col, mapping, new_col=None):
    """
    Reclassify categories of a categorical column.

    Parameters:
    ----------
    data : pd.DataFrame
        Input dataset.
    col : str
        Column name to reclassify.
    mapping : dict
        Dictionary mapping old categories to new categories.
        Example: {"Bachelor": "HigherEdu", "Master": "HigherEdu", "PhD": "HigherEdu"}
    new_col : str, optional
        Name of new column. If None, overwrites the original column.

    Returns:
    -------
    pd.DataFrame
        DataFrame with reclassified column.
    """
    if new_col is None:
        new_col = col
    
    data[new_col] = data[col].replace(mapping)
    return data
    

## Grouping Occupation and Therapy History

In this step, we further simplify and standardize categorical variables by grouping similar categories into broader classes.

This helps to:
- Reduce high cardinality in categorical features
- Improve interpretability of the dataset
- Make features more suitable for encoding and modeling
- Capture higher-level patterns in the data

### Occupation Grouping

Occupations are grouped into meaningful categories such as:
- Academic
- Healthcare
- Technical
- Creative
- Professional
- Other

### Therapy History Grouping

Therapy-related categories are grouped into:
- Therapy
- No Therapy

This ensures consistency between different therapy-related labels and simplifies analysis.

In [ ]:
occupation_map = {
    "Teacher": "Academic",
    "Student": "Academic",
    "Scientist": "Academic",

    "Doctor": "Healthcare",
    "Nurse": "Healthcare",

    "Engineer": "Technical",

    "Artist": "Creative",
    "Musician": "Creative",
    "Chef": "Creative",

    "Lawyer": "Professional",
    "Athlete": "Professional",

    "Freelancer": "Other"
   
}

df = reclassify_categorical(df, "Occupation", occupation_map, new_col="Occupation_Group")


In [ ]:
therapy_map = {
    "Psychodynamic Therapy": "Therapy",
    "Cognitive Behavioral Therapy (CBT)": "Therapy",
    "Group Therapy": "Therapy",
    "No previous history": "No Therapy",
    "Not Reported": "Therapy"
    
}

df = reclassify_categorical(df, "Therapy History", therapy_map, new_col="Therapy_Group")

In [ ]:
df

,Age,Gender,Occupation,Sleep Hours,Physical Activity (hrs/week),Caffeine Intake (mg/day),Alcohol Consumption (drinks/week),Smoking,Family History of Anxiety,Stress Level (1-10),...,Binned_Caffeine Intake (mg/day),Binned_Alcohol Consumption (drinks/week),Binned_Heart Rate (bpm),Binned_Breathing Rate (breaths/min),Binned_Therapy Sessions (per month),Binned_Stress Level (1-10),Binned_Diet Quality (1-10),Binned_Anxiety Level (1-10),Occupation_Group,Therapy_Group
0,59.0,Other,Teacher,7.0,2.4,40.0,5.000000,Yes,No,4.0,...,Low,Light,Normal,Rapid,No Therapy,Moderate Stress,Unhealthy,Low Anxiety,Academic,No Therapy
1,46.0,Female,Student,5.1,5.4,156.0,11.000000,Yes,No,3.0,...,Moderate,Heavy,Normal,Normal,Occasional,Low Stress,Unhealthy,Moderate Anxiety,Academic,Therapy
2,40.0,Other,Lawyer,5.1,1.9,570.0,14.000000,Yes,Yes,9.0,...,High,Heavy,High,Rapid,Intensive,High Stress,Average,High Anxiety,Professional,Therapy
3,40.0,Male,Nurse,7.6,0.9,129.0,0.000000,No,No,9.0,...,Moderate,None,High,Normal,Occasional,High Stress,Unhealthy,Moderate Anxiety,Healthcare,Therapy
4,26.0,Male,Other,6.7,3.0,64.0,13.000000,No,No,15.0,...,Low,Heavy,High,Normal,No Therapy,NaN,Average,Low Anxiety,Other,No Therapy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025,42.0,Male,Teacher,3.7,2.9,113.0,18.000000,Yes,Yes,4.0,...,Moderate,Heavy,Normal,Normal,Occasional,Moderate Stress,Unhealthy,Low Anxiety,Academic,Therapy
2026,34.0,Other,Freelancer,8.4,2.0,239.0,14.000000,No,No,7.0,...,Moderate,Heavy,Normal,Normal,No Therapy,High Stress,Average,Moderate Anxiety,Other,No Therapy
2027,62.0,Female,Musician,6.1,1.6,149.0,18.000000,Yes,No,2.0,...,Moderate,Heavy,Normal,Rapid,Regular,Low Stress,Healthy,Low Anxiety,Creative,Therapy
2028,47.0,Other,Nurse,6.3,1.1,298.0,9.786432,No,Yes,7.0,...,Moderate,Moderate,Normal,Rapid,Occasional,High Stress,Unhealthy,Moderate Anxiety,Healthcare,Therapy


In [ ]:
def create_kpi_features(df):

    df = df.copy()

  
    df["KPI_Anxiety_Risk_Index"] = (

        (
            df["Stress Level (1-10)"]
            *
            df["Anxiety Level (1-10)"]
        )

        /

        (
            df["Sleep Hours"]
            +
            df["Physical Activity (hrs/week)"]
            +
             df["Diet Quality (1-10)"]
        )
    )

  
    df["KPI_Therapy_Gap"] = (

        df["Anxiety Level (1-10)"]

        -

        df["Therapy Sessions (per month)"]

    )

  


 
    df["KPI_Lifestyle_Risk"] = (

        (
            df["Sleep Hours"] < 6
        ).astype(int)

        +

        (
            df["Physical Activity (hrs/week)"] < 3
        ).astype(int)

        +

        (
            df["Diet Quality (1-10)"] < 5
        ).astype(int)

        +

        (
            df["Alcohol Consumption (drinks/week)"] > 7
        ).astype(int)

        +

        (
            df["Smoking"] == "Yes"
        ).astype(int)

    )
    df["KPI_Mental_Health_Risk_Score"] = (
    (
        df["Stress Level (1-10)"] +
        df["Anxiety Level (1-10)"]
    )
    +
    (
        df["Sleep Hours"].apply(lambda x: max(0, 8 - x)) +
        df["Physical Activity (hrs/week)"].apply(lambda x: max(0, 5 - x))
    )
    +
    (
        df["Diet Quality (1-10)"].apply(lambda x: max(0, 6 - x))
    )
    +
    (
        df["Smoking"] == "Yes"
    ).astype(int)
    +
    (
        df["Alcohol Consumption (drinks/week)"].apply(lambda x: max(0, x - 7))
    )
)

    return df

In [ ]:
df=create_kpi_features(df)

## Saving Cleaned Dataset

In this final step, we save the processed and cleaned dataset into a new CSV file.

This allows us to:
- Export the cleaned version of the data
- Share it with team members for further analysis
- Use it directly in modeling or dashboarding without repeating preprocessing steps

The `index=False` parameter ensures that the DataFrame index is not saved as an extra column.

In [ ]:
df.to_csv("cleaned_data.csv", index=False)

In [ ]:
data=pd.read_csv("cleaned_data.csv")